In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('processed_data')
PROCESSED_DIR.mkdir(exist_ok=True)

print('Setup selesai.')

In [ ]:
segmen_dfs = {}
for f in sorted(PROCESSED_DIR.glob('segmen_*.csv')):
    sid = int(f.stem.split('_')[1])
    segmen_dfs[sid] = pd.read_csv(f)
    segmen_dfs[sid]['datetime'] = pd.to_datetime(segmen_dfs[sid]['datetime'])

print(f'{len(segmen_dfs)} segmen loaded:')
for sid, seg in segmen_dfs.items():
    print(f'  segmen_{sid}: {len(seg)} rows ({seg["datetime"].iloc[0]} s/d {seg["datetime"].iloc[-1]})')

In [ ]:
for sid in list(segmen_dfs.keys()):
    df = segmen_dfs[sid].sort_values('datetime').reset_index(drop=True)

    # Lag PM25: 1,2,3,4,12,24,48,72,124
    for lag in [1, 2, 3, 4, 12, 24, 48, 72, 124]:
        df[f'pm25_lag_{lag}h'] = df['pm25'].shift(lag)

    # Lag PM1: 1
    df['pm1_lag_1h'] = df['pm1'].shift(1)

    # Lag um003: 1
    df['um003_lag_1h'] = df['um003'].shift(1)

    # Lag RH: 6,12,24
    for lag in [6, 12, 24]:
        df[f'relativehumidity_lag_{lag}h'] = df['relativehumidity'].shift(lag)

    # Lag Temp: 6,12,24
    for lag in [6, 12, 24]:
        df[f'temperature_lag_{lag}h'] = df['temperature'].shift(lag)

    # Rolling Mean PM25: 24,48,72
    for w in [24, 48, 72]:
        df[f'pm25_rolling_mean_{w}h'] = df['pm25'].shift(1).rolling(w, min_periods=1).mean()

    # Diff PM25: 1, 24
    df['pm25_diff_1h'] = df['pm25'].diff(1)
    df['pm25_diff_24h'] = df['pm25'].diff(24)

    # Calendar features
    dt = df['datetime']
    df['hour'] = dt.dt.hour
    df['day_of_week'] = dt.dt.dayofweek
    df['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)

    # Cyclic hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    segmen_dfs[sid] = df

print(f'Feature engineering selesai untuk {len(segmen_dfs)} segmen.')

In [ ]:
df_all = pd.concat(segmen_dfs.values(), ignore_index=True)
df_all = df_all.sort_values('datetime').reset_index(drop=True)
print(f'Gabungan: {len(df_all)} rows, {df_all["datetime"].iloc[0]} s/d {df_all["datetime"].iloc[-1]}')

In [ ]:
n_before = len(df_all)
df_all = df_all.dropna().reset_index(drop=True)
n_dropped = n_before - len(df_all)
print(f'Dropped {n_dropped} rows ({n_dropped/n_before*100:.1f}%), tersisa {len(df_all)} rows.')

In [ ]:
target = 'pm25'
exclude = {'datetime', 'pm25'}
feature_cols = [c for c in df_all.columns if c not in exclude]

split_idx = int(len(df_all) * 0.8)
X_train = df_all[feature_cols].iloc[:split_idx]
X_test  = df_all[feature_cols].iloc[split_idx:]
y_train = df_all[target].iloc[:split_idx]
y_test  = df_all[target].iloc[split_idx:]

print(f'Train: {len(X_train)} rows')
print(f'Test:  {len(X_test)} rows')
print(f'Features: {len(feature_cols)}')

In [ ]:
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

import json
with open(PROCESSED_DIR / 'feature_columns.json', 'w') as f:
    json.dump({'feature_columns': feature_cols, 'target': target}, f, indent=2)

print('Semua file tersimpan di processed_data/:')                                        
for f in sorted(PROCESSED_DIR.glob('*')):
    print(f'  {f.name}')